# Configuración de LlamaIndex

In [4]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

llm_model = OpenAI(
    model="gpt-4o-mini",
    api_key=api_key
)

llm_embeddings = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=api_key
)

Settings.llm = llm_model
Settings.embed_model = llm_embeddings

# Tools

In [5]:
import requests

def mensaje_por_telegram(mensaje):
    """Útil cuando se solicita enviar un MENSAJE a Telegram"""

    url_getIdChat = 'https://api.telegram.org/bot'+bot_token+'/getUpdates'
    response = requests.get(url_getIdChat)

    bot_message = mensaje
    url_sendMessage = 'https://api.telegram.org/bot'+bot_token+'/sendMessage?chat_id='+bot_chatId+'&parse_mode=Markdown&text='+bot_message

    response = requests.get(url_sendMessage)
    return response.text

In [6]:
mensaje_por_telegram(mensaje="prueba-notebook")

'{"ok":true,"result":{"message_id":272,"from":{"id":1060305894,"is_bot":true,"first_name":"\\ud83c\\udfc1script","username":"edustratiobot"},"chat":{"id":125821294,"first_name":"Eduardo","last_name":"FC","type":"private"},"date":1775293813,"text":"prueba-notebook"}}'

In [7]:
from llama_index.core.tools import FunctionTool

telegram_tool = FunctionTool.from_defaults(
    mensaje_por_telegram
)

# ReAct Agent

In [8]:
from llama_index.core.agent import ReActAgent

agent = ReActAgent.from_tools(
    [telegram_tool], verbose=True
)

In [9]:
user_query = "enviame un mensaje por telegram que diga: INCREIBLE QUE UN LLM PUEDA ENVIAR UN MENSAJE AL EXTERIOR"

In [10]:
response = agent.chat(user_query)
response

> Running step f0114691-00d9-46be-a166-086d14d830b0. Step input: enviame un mensaje por telegram que diga: INCREIBLE QUE UN LLM PUEDA ENVIAR UN MENSAJE AL EXTERIOR
Thought: The current language of the user is: Spanish. I need to use a tool to help me answer the question.
Action: mensaje_por_telegram
Action Input: {'mensaje': 'INCREIBLE QUE UN LLM PUEDA ENVIAR UN MENSAJE AL EXTERIOR'}
Observation: {"ok":true,"result":{"message_id":273,"from":{"id":1060305894,"is_bot":true,"first_name":"\ud83c\udfc1script","username":"edustratiobot"},"chat":{"id":125821294,"first_name":"Eduardo","last_name":"FC","type":"private"},"date":1775294166,"text":"INCREIBLE QUE UN LLM PUEDA ENVIAR UN MENSAJE AL EXTERIOR"}}
> Running step faf0d8bc-601d-4185-b942-8a945138738f. Step input: None
Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: El mensaje ha sido enviado correctamente por Telegram.


AgentChatResponse(response='El mensaje ha sido enviado correctamente por Telegram.', sources=[ToolOutput(content='{"ok":true,"result":{"message_id":273,"from":{"id":1060305894,"is_bot":true,"first_name":"\\ud83c\\udfc1script","username":"edustratiobot"},"chat":{"id":125821294,"first_name":"Eduardo","last_name":"FC","type":"private"},"date":1775294166,"text":"INCREIBLE QUE UN LLM PUEDA ENVIAR UN MENSAJE AL EXTERIOR"}}', tool_name='mensaje_por_telegram', raw_input={'args': (), 'kwargs': {'mensaje': 'INCREIBLE QUE UN LLM PUEDA ENVIAR UN MENSAJE AL EXTERIOR'}}, raw_output='{"ok":true,"result":{"message_id":273,"from":{"id":1060305894,"is_bot":true,"first_name":"\\ud83c\\udfc1script","username":"edustratiobot"},"chat":{"id":125821294,"first_name":"Eduardo","last_name":"FC","type":"private"},"date":1775294166,"text":"INCREIBLE QUE UN LLM PUEDA ENVIAR UN MENSAJE AL EXTERIOR"}}', is_error=False)], source_nodes=[], is_dummy_stream=False, metadata=None)

In [12]:
# response = agent.chat("cuales son las coberturas en caso de romper un cristal")
# response

In [16]:
print(agent.get_prompts()['agent_worker:system_prompt'].template)

You are designed to help with a variety of tasks, from answering questions to providing summaries to other types of analyses.

## Tools

You have access to a wide variety of tools. You are responsible for using the tools in any sequence you deem appropriate to complete the task at hand.
This may require breaking the task into subtasks and using different tools to complete each subtask.

You have access to the following tools:
{tool_desc}


## Output Format

Please answer in the same language as the question and use the following format:

```
Thought: The current language of the user is: (user's language). I need to use a tool to help me answer the question.
Action: tool name (one of {tool_names}) if using a tool.
Action Input: the input to the tool, in a JSON format representing the kwargs (e.g. {{"input": "hello world", "num_beams": 5}})
```

Please ALWAYS start with a Thought.

NEVER surround your response with markdown code markers. You may use code markers within your response if y

In [18]:
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core.tools import QueryEngineTool, ToolMetadata

sc = StorageContext.from_defaults(persist_dir='vector_store')
index = load_index_from_storage(sc)
qe = index.as_query_engine()

rag_tool = QueryEngineTool(
    query_engine=qe,
    metadata=ToolMetadata(
        name="Seguros",
        description="Útil cuando hay que dar respuesta a preguntas relacionadas con Seguro de hogar y sus coberturas."
    )
)

In [19]:
agent2 = ReActAgent.from_tools(
    [telegram_tool, rag_tool], verbose=True
)

In [28]:
# user_query = "Tengo alguna cobertura que cubra humos en mi seguro de hogar?"
# user_query = "enviame un mensaje por telegram que diga: INCREIBLE QUE UN LLM PUEDA ENVIAR UN MENSAJE AL EXTERIOR"
user_query = "Tengo alguna cobertura que cubra humos en mi seguro de hogar? Si es asi, envíamela por Telegram."

In [29]:
agent2.chat(user_query)

> Running step cadea29a-db97-4acc-aa96-e533d8cb39b8. Step input: Tengo alguna cobertura que cubra humos en mi seguro de hogar? Si es asi, envíamela por Telegram.
Thought: El usuario quiere saber si su seguro de hogar cubre daños por humo y que esta información sea enviada por Telegram. Necesito usar la herramienta de Seguros para obtener la información sobre la cobertura de humos.
Action: Seguros
Action Input: {'input': 'Tengo alguna cobertura que cubra humos en mi seguro de hogar?'}
Observation: Sí, tu seguro de hogar cubre los daños ocasionados por el humo, incluyendo aquellos que provienen de escapes repentinos en cocinas, sistemas de calefacción y otros aparatos eléctricos. Esto incluye la reparación de daños como la limpieza de paredes ennegrecidas por humo.
> Running step 0b1d7712-0426-4d43-af00-0fa18f1e371b. Step input: None
Thought: Puedo enviar la información sobre la cobertura de humos en el seguro de hogar por Telegram.
Action: mensaje_por_telegram
Action Input: {'mensaje': 

AgentChatResponse(response='La información ha sido enviada correctamente. Si necesitas más ayuda, no dudes en preguntar.', sources=[ToolOutput(content='Sí, tu seguro de hogar cubre los daños ocasionados por el humo, incluyendo aquellos que provienen de escapes repentinos en cocinas, sistemas de calefacción y otros aparatos eléctricos. Esto incluye la reparación de daños como la limpieza de paredes ennegrecidas por humo.', tool_name='Seguros', raw_input={'input': 'Tengo alguna cobertura que cubra humos en mi seguro de hogar?'}, raw_output=Response(response='Sí, tu seguro de hogar cubre los daños ocasionados por el humo, incluyendo aquellos que provienen de escapes repentinos en cocinas, sistemas de calefacción y otros aparatos eléctricos. Esto incluye la reparación de daños como la limpieza de paredes ennegrecidas por humo.', source_nodes=[NodeWithScore(node=TextNode(id_='462a5625-576f-404f-b03b-b90c3d93ecf3', embedding=None, metadata={'page_label': '2', 'file_name': 'AF-SL-TablasGarant

# OpenAI Agent

In [30]:
from llama_index.agent.openai import OpenAIAgent

agent_openai = OpenAIAgent.from_tools(
    [telegram_tool], verbose=True,
)

In [31]:
response = agent_openai.chat("Enviame un 'hola mundo' por telegram")
response

Added user message to memory: Enviame un 'hola mundo' por telegram
=== Calling Function ===
Calling function: mensaje_por_telegram with args: {"mensaje":"hola mundo"}
Got output: {"ok":true,"result":{"message_id":278,"from":{"id":1060305894,"is_bot":true,"first_name":"\ud83c\udfc1script","username":"edustratiobot"},"chat":{"id":125821294,"first_name":"Eduardo","last_name":"FC","type":"private"},"date":1775295439,"text":"hola mundo"}}



AgentChatResponse(response='He enviado el mensaje "hola mundo" por Telegram.', sources=[ToolOutput(content='{"ok":true,"result":{"message_id":278,"from":{"id":1060305894,"is_bot":true,"first_name":"\\ud83c\\udfc1script","username":"edustratiobot"},"chat":{"id":125821294,"first_name":"Eduardo","last_name":"FC","type":"private"},"date":1775295439,"text":"hola mundo"}}', tool_name='mensaje_por_telegram', raw_input={'args': (), 'kwargs': {'mensaje': 'hola mundo'}}, raw_output='{"ok":true,"result":{"message_id":278,"from":{"id":1060305894,"is_bot":true,"first_name":"\\ud83c\\udfc1script","username":"edustratiobot"},"chat":{"id":125821294,"first_name":"Eduardo","last_name":"FC","type":"private"},"date":1775295439,"text":"hola mundo"}}', is_error=False)], source_nodes=[], is_dummy_stream=False, metadata=None)